# 08. Fine-tune `BAAI/bge-m3` на русских парафразах

Гипотеза: лёгкая донастройка текущего векторизатора `BAAI/bge-m3` на русских парафразах может лучше сближать рерайты и перепечатки, а значит потенциально улучшить story clustering и downstream novelty detection.

Важно: метрики на парафразах — это только sanity-check. Главная проверка ниже — downstream smoke-test на `lenta_golden_candidate_pool.csv`: `pairwise_f1`, `false_merge_rate`, количество рёбер и кластеров. Если smoke-test выглядит перспективно, полноценный эксперимент нужно переносить в `06_model_improvement_experiments.ipynb`.

## 1. Импорты, пути и конфиг

In [1]:
# !pip install -U datasets sentence-transformers accelerate pandas scikit-learn

from pathlib import Path
import sys
import json
import random
from time import perf_counter

import numpy as np
import pandas as pd

from datasets import load_dataset, load_from_disk, Dataset


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / ".git").exists() and (path / "README.md").exists():
            return path
    raise RuntimeError("Project root not found")


PROJECT_DIR = find_project_root()
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PREPARED_DIR = DATA_DIR / "prepared"
ARTIFACTS_DIR = DATA_DIR / "artifacts"
MODELS_DIR = PROJECT_DIR / "models"

for directory in [RAW_DIR, PREPARED_DIR, ARTIFACTS_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Dataset
DATASET_NAME = "merionum/ru_paraphraser"
LOCAL_DATASET_PATH = RAW_DIR / "ru_paraphraser_hf"

# Model
BASE_MODEL_NAME = "BAAI/bge-m3"
EXPERIMENT_SLUG = "news-flow-bge-m3-ru-paraphrase-mnrl"
OUTPUT_DIR = MODELS_DIR / EXPERIMENT_SLUG
FINETUNED_MODEL_PATH = OUTPUT_DIR / "final"

# Training switches
DO_TRAIN = True
FORCE_RETRAIN = False

# Conservative params for RTX 4070 12 GB
NUM_TRAIN_EPOCHS = 1
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-5
WARMUP_RATIO = 0.10
FP16 = True
MAX_SEQ_LENGTH = 512

# Evaluation
RETRIEVAL_BATCH_SIZE = 16
DOWNSTREAM_ENCODE_BATCH_SIZE = 16

print("PROJECT_DIR:", PROJECT_DIR)
print("BASE_MODEL_NAME:", BASE_MODEL_NAME)
print("FINETUNED_MODEL_PATH:", FINETUNED_MODEL_PATH)

PROJECT_DIR: E:\ML\Projects\Git\news-flow-analysis
BASE_MODEL_NAME: BAAI/bge-m3
FINETUNED_MODEL_PATH: E:\ML\Projects\Git\news-flow-analysis\models\news-flow-bge-m3-ru-paraphrase-mnrl\final


In [ ]:
# ---------------------------------------------------------------------
# Safer output dirs for BGE-M3 fine-tuning
# ---------------------------------------------------------------------

from pathlib import Path
import shutil
import gc
import torch

# Не сохраняем большие checkpoint'ы внутрь Git/project dir.
# Лучше отдельная локальная папка вне синхронизации.
TRAINING_RUN_DIR = Path(r"E:\MLCache\news-flow-analysis\runs\bge_m3_ru_paraphrase_mnrl")
FINAL_MODEL_DIR = Path(r"E:\MLCache\news-flow-analysis\models\bge_m3_ru_paraphrase_mnrl")

# Если предыдущий запуск упал на сохранении — удаляем битый run dir.
if TRAINING_RUN_DIR.exists():
    shutil.rmtree(TRAINING_RUN_DIR)

TRAINING_RUN_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("TRAINING_RUN_DIR:", TRAINING_RUN_DIR)
print("FINAL_MODEL_DIR:", FINAL_MODEL_DIR)

## 2. Загрузка и подготовка `ru_paraphraser`

In [2]:
if LOCAL_DATASET_PATH.exists():
    dataset = load_from_disk(str(LOCAL_DATASET_PATH))
    print(f"Loaded from disk: {LOCAL_DATASET_PATH}")
else:
    dataset = load_dataset(DATASET_NAME)
    dataset.save_to_disk(str(LOCAL_DATASET_PATH))
    print(f"Downloaded and saved to: {LOCAL_DATASET_PATH}")

print(dataset)
for split_name in dataset.keys():
    print("\nSPLIT:", split_name)
    print("Rows:", len(dataset[split_name]))
    print("Columns:", dataset[split_name].column_names)
    print(dataset[split_name][0])

Saving the dataset (0/1 shards):   0%|          | 0/7227 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1924 [00:00<?, ? examples/s]

Downloaded and saved to: E:\ML\Projects\Git\news-flow-analysis\data\raw\ru_paraphraser_hf
DatasetDict({
    train: Dataset({
        features: ['id', 'id_1', 'id_2', 'text_1', 'text_2', 'class'],
        num_rows: 7227
    })
    test: Dataset({
        features: ['id', 'id_1', 'id_2', 'text_1', 'text_2', 'class'],
        num_rows: 1924
    })
})

SPLIT: train
Rows: 7227
Columns: ['id', 'id_1', 'id_2', 'text_1', 'text_2', 'class']
{'id': '1', 'id_1': '201', 'id_2': '8159', 'text_1': 'Полицейским разрешат стрелять на поражение по гражданам с травматикой.', 'text_2': 'Полиции могут разрешить стрелять по хулиганам с травматикой.', 'class': '0'}

SPLIT: test
Rows: 1924
Columns: ['id', 'id_1', 'id_2', 'text_1', 'text_2', 'class']
{'id': '25349', 'id_1': '34420', 'id_2': '34421', 'text_1': 'Цены на нефть восстанавливаются', 'text_2': 'Парламент Словакии поблагодарил народы бывшего СССР за Победу', 'class': '-1'}


In [3]:
def detect_columns(df: pd.DataFrame):
    cols = set(df.columns)
    possible_text1 = ["text_1", "text1", "sentence1", "sentence_1", "s1"]
    possible_text2 = ["text_2", "text2", "sentence2", "sentence_2", "s2"]
    possible_label = ["label", "class", "target"]

    text1_col = next((c for c in possible_text1 if c in cols), None)
    text2_col = next((c for c in possible_text2 if c in cols), None)
    label_col = next((c for c in possible_label if c in cols), None)

    if text1_col is None or text2_col is None or label_col is None:
        raise ValueError(f"Could not detect columns. Available columns: {df.columns.tolist()}")

    return text1_col, text2_col, label_col


train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas() if "test" in dataset else None
TEXT1_COL, TEXT2_COL, LABEL_COL = detect_columns(train_df)

train_df[LABEL_COL] = train_df[LABEL_COL].astype(int)
if test_df is not None:
    test_df[LABEL_COL] = test_df[LABEL_COL].astype(int)

positive_df = train_df[train_df[LABEL_COL] == 1].copy()
positive_df = positive_df[[TEXT1_COL, TEXT2_COL, LABEL_COL]].dropna()
positive_df[TEXT1_COL] = positive_df[TEXT1_COL].astype(str).str.strip()
positive_df[TEXT2_COL] = positive_df[TEXT2_COL].astype(str).str.strip()
positive_df = positive_df[
    (positive_df[TEXT1_COL].str.len() > 0)
    & (positive_df[TEXT2_COL].str.len() > 0)
].reset_index(drop=True)

prepared_pairs_path = PREPARED_DIR / "ru_paraphraser_positive_pairs.parquet"
positive_df.to_parquet(prepared_pairs_path, index=False)

print("Text columns:", TEXT1_COL, TEXT2_COL)
print("Label column:", LABEL_COL)
print("Train labels:")
print(train_df[LABEL_COL].value_counts(dropna=False))
print("Positive pairs:", len(positive_df))
display(positive_df.head())

Text columns: text_1 text_2
Label column: class
Train labels:
class
 0    2957
-1    2582
 1    1688
Name: count, dtype: int64
Positive pairs: 1688


,text_1,text_2,class
0,Приставы соберут отпечатки пальцев российских ...,Приставы снимут отпечатки пальцев у злостных н...,1
1,Москвичи смогут забронировать в Интернете мест...,В Москве можно будет забронировать место на кл...,1
2,Северокорейский лидер впервые за 19 лет поздра...,Лидер КНДР впервые за 19 лет поздравил согражд...,1
3,Мужчина из Подмосковья случайно убил жену в Но...,Житель Подмосковья случайно убил жену на новог...,1
4,Житель Украины расстрелял посетителей кафе.,На Украине мужчина через окно расстрелял посет...,1


In [4]:
from sklearn.model_selection import train_test_split

train_pairs_df, eval_pairs_df = train_test_split(
    positive_df,
    test_size=0.10,
    random_state=RANDOM_STATE,
    shuffle=True,
)

train_pairs = Dataset.from_pandas(
    train_pairs_df[[TEXT1_COL, TEXT2_COL]].rename(
        columns={TEXT1_COL: "sentence1", TEXT2_COL: "sentence2"}
    ),
    preserve_index=False,
)

eval_pairs = Dataset.from_pandas(
    eval_pairs_df[[TEXT1_COL, TEXT2_COL]].rename(
        columns={TEXT1_COL: "sentence1", TEXT2_COL: "sentence2"}
    ),
    preserve_index=False,
)

print("train_pairs_df:", train_pairs_df.shape)
print("eval_pairs_df:", eval_pairs_df.shape)
print(train_pairs)

train_pairs_df: (1519, 3)
eval_pairs_df: (169, 3)
Dataset({
    features: ['sentence1', 'sentence2'],
    num_rows: 1519
})


## 3. Fine-tune BGE-M3 через `MultipleNegativesRankingLoss`

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer

if DO_TRAIN:
    if FINETUNED_MODEL_PATH.exists() and not FORCE_RETRAIN:
        print(f"Fine-tuned model already exists: {FINETUNED_MODEL_PATH}")
        print("Set FORCE_RETRAIN=True to retrain.")
    else:
        start = perf_counter()

        model = SentenceTransformer(BASE_MODEL_NAME)
        model.max_seq_length = MAX_SEQ_LENGTH
        loss = MultipleNegativesRankingLoss(model)

        args = SentenceTransformerTrainingArguments(
            output_dir=str(TRAINING_RUN_DIR),

            num_train_epochs=1,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            learning_rate=1e-5,
            warmup_ratio=0.1,

            fp16=True,
            bf16=False,

            # ВАЖНО: не пишем промежуточные checkpoint'ы.
            save_strategy="no",

            logging_steps=25,
            report_to="none",

            dataloader_num_workers=0,
        )

        trainer = SentenceTransformerTrainer(
            model=model,
            args=args,
            train_dataset=train_pairs,
            loss=loss,
        )

        trainer.train()

        # Сохраняем только финальную модель.
        try:
            model.save_pretrained(str(FINAL_MODEL_DIR), safe_serialization=True)
        except TypeError:
            # fallback для старых версий sentence-transformers
            model.save(str(FINAL_MODEL_DIR))

        print("Saved fine-tuned model to:", FINAL_MODEL_DIR)

        elapsed = perf_counter() - start
        print(f"Saved model to: {FINETUNED_MODEL_PATH}")
        print(f"Training time: {elapsed / 60:.2f} min")
else:
    print("DO_TRAIN=False, training skipped.")

FINE_TUNED_MODEL_PATH = FINAL_MODEL_DIR

C:\Users\lazer\AppData\Local\Temp\ipykernel_7956\2571530784.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss
C:\Users\lazer\AppData\Local\Temp\ipykernel_7956\2571530784.py:3: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import SentenceTransformerTrainingArguments
C:\Users\lazer\AppData\Local\Temp\ipykernel_7956\2571530784.py:4: DeprecationWarning: Importing from 'sentence_transformers.trainer' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.trainer' instead.
  from sentence_transformers.trainer import Sent

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
25,0.011916
50,0.000201
75,0.000156


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: [enforce fail at inline_container.cc:672] . unexpected pos 1024115072 vs 1024114960

In [ ]:
metadata = {
    "base_model_name": BASE_MODEL_NAME,
    "experiment_slug": EXPERIMENT_SLUG,
    "finetuned_model_path": str(FINETUNED_MODEL_PATH),
    "dataset_name": DATASET_NAME,
    "positive_pairs": int(len(positive_df)),
    "train_pairs": int(len(train_pairs_df)),
    "eval_pairs": int(len(eval_pairs_df)),
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "fp16": FP16,
    "max_seq_length": MAX_SEQ_LENGTH,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metadata_path = OUTPUT_DIR / "experiment_metadata.json"
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print(metadata_path)

## 4. Sanity-check: retrieval на парафразах

In [ ]:
base_model = SentenceTransformer(BASE_MODEL_NAME)
base_model.max_seq_length = MAX_SEQ_LENGTH

if not FINETUNED_MODEL_PATH.exists():
    raise FileNotFoundError(f"Fine-tuned model not found: {FINETUNED_MODEL_PATH}")

ft_model = SentenceTransformer(str(FINETUNED_MODEL_PATH))
ft_model.max_seq_length = MAX_SEQ_LENGTH

queries = eval_pairs_df[TEXT1_COL].astype(str).tolist()
positives = eval_pairs_df[TEXT2_COL].astype(str).tolist()

negatives_pool = train_df[train_df[LABEL_COL] == -1][TEXT2_COL].astype(str).dropna().tolist()
random.seed(RANDOM_STATE)
extra_negatives = random.sample(negatives_pool, min(2000, len(negatives_pool)))

candidates = positives + extra_negatives
seen = set()
unique_candidates = []
for text in candidates:
    if text not in seen:
        unique_candidates.append(text)
        seen.add(text)

candidates = unique_candidates
candidate_index = {text: i for i, text in enumerate(candidates)}
target_indices = [candidate_index[p] for p in positives]

print("Queries:", len(queries))
print("Candidates:", len(candidates))

In [ ]:
def evaluate_retrieval_numpy(model, queries, candidates, target_indices, top_k=10, batch_size=16):
    query_emb = model.encode(
        queries,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")

    candidate_emb = model.encode(
        candidates,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")

    scores = query_emb @ candidate_emb.T

    recall_at_1 = recall_at_5 = recall_at_10 = 0
    mrr = 0.0

    for row_idx, row_scores in enumerate(scores):
        top_indices = np.argsort(-row_scores)[:top_k]
        target = target_indices[row_idx]

        recall_at_1 += int(target == top_indices[0])
        recall_at_5 += int(target in top_indices[:5])
        recall_at_10 += int(target in top_indices[:10])

        rank_positions = np.where(top_indices == target)[0]
        if len(rank_positions) > 0:
            mrr += 1.0 / (int(rank_positions[0]) + 1)

    n = len(queries)
    return {
        "Recall@1": recall_at_1 / n,
        "Recall@5": recall_at_5 / n,
        "Recall@10": recall_at_10 / n,
        "MRR@10": mrr / n,
    }

In [ ]:
base_retrieval_metrics = evaluate_retrieval_numpy(
    base_model, queries, candidates, target_indices,
    top_k=10, batch_size=RETRIEVAL_BATCH_SIZE,
)

ft_retrieval_metrics = evaluate_retrieval_numpy(
    ft_model, queries, candidates, target_indices,
    top_k=10, batch_size=RETRIEVAL_BATCH_SIZE,
)

retrieval_table = pd.DataFrame(
    [base_retrieval_metrics, ft_retrieval_metrics],
    index=["base_bge_m3", "finetuned_bge_m3"],
)

retrieval_table

## 5. Downstream smoke-test: clustering на golden candidate pool

Проверяем только кластеризацию, без final novelty classifier. Это быстрый фильтр: если fine-tuned embeddings ухудшили `false_merge_rate` или `pairwise_f1`, переносить их в `06` не стоит.

In [ ]:
CANDIDATE_POOL_PATH = PREPARED_DIR / "lenta_golden_candidate_pool.csv"
GOLDEN_PATH = PREPARED_DIR / "lenta_golden_set_human.csv"

print("CANDIDATE_POOL_PATH:", CANDIDATE_POOL_PATH, CANDIDATE_POOL_PATH.exists())
print("GOLDEN_PATH:", GOLDEN_PATH, GOLDEN_PATH.exists())

In [ ]:
def normalize_news_id(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)


def prepare_model_input(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = raw_df.copy()
    df["news_id"] = normalize_news_id(df["news_id"])
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    if df["published_at"].isna().any():
        raise ValueError(f"Invalid published_at rows: {df['published_at'].isna().sum()}")

    df["title"] = df["title"].fillna("").astype(str)
    df["text"] = df["text"].fillna("").astype(str)

    if "model_text" not in df.columns:
        df["model_text"] = df["title"] + "\n\n" + df["text"]
    else:
        df["model_text"] = df["model_text"].fillna("").astype(str)

    df["_news_id_sort_key"] = pd.to_numeric(df["news_id"], errors="coerce")
    df = (
        df
        .sort_values(["published_at", "_news_id_sort_key", "news_id"], kind="mergesort")
        .drop(columns=["_news_id_sort_key"])
        .reset_index(drop=True)
    )
    return df


def get_or_create_embeddings(model, news_df: pd.DataFrame, cache_path: Path, batch_size: int = 16, force_recompute: bool = False) -> np.ndarray:
    ids = normalize_news_id(news_df["news_id"]).to_numpy()

    if cache_path.exists() and not force_recompute:
        cache = np.load(cache_path, allow_pickle=True)
        if "ids" in cache and "embeddings" in cache:
            cached_ids = cache["ids"].astype(str)
            embeddings = cache["embeddings"].astype("float32")
            id_to_pos = {news_id: idx for idx, news_id in enumerate(cached_ids)}
            missing = [news_id for news_id in ids if news_id not in id_to_pos]
            if not missing:
                aligned = np.vstack([embeddings[id_to_pos[news_id]] for news_id in ids]).astype("float32")
                print(f"Loaded id-aligned embeddings from cache: {cache_path}")
                return aligned
            print(f"Cache exists but has missing ids: {len(missing)}. Recomputing.")

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    embeddings = model.encode(
        news_df["model_text"].astype(str).tolist(),
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")

    np.savez_compressed(cache_path, ids=ids.astype(str), embeddings=embeddings)
    print(f"Saved embeddings cache: {cache_path}")
    return embeddings

In [ ]:
class DSU:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int) -> bool:
        ra = self.find(a)
        rb = self.find(b)
        if ra == rb:
            return False
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        return True


def threshold_graph_cluster(news_df, embeddings, story_threshold=0.82, story_window_days=14, use_topic_blocking=True):
    news = news_df.copy().reset_index(drop=True)
    news["published_at"] = pd.to_datetime(news["published_at"], errors="coerce")
    embeddings = np.asarray(embeddings, dtype="float32")

    if len(news) != len(embeddings):
        raise ValueError(f"length mismatch: {len(news)} != {len(embeddings)}")

    dsu = DSU(len(news))
    edge_count = 0

    if use_topic_blocking and "topic" in news.columns:
        groups = news.groupby("topic", sort=False).groups.values()
    else:
        groups = [np.arange(len(news))]

    for group_indices in groups:
        group_indices = np.array(list(group_indices), dtype=int)
        group_df = news.iloc[group_indices].sort_values("published_at", kind="mergesort")
        sorted_indices = group_df.index.to_numpy(dtype=int)
        dates = news.loc[sorted_indices, "published_at"].to_numpy()

        for pos, current_idx in enumerate(sorted_indices):
            current_date = pd.Timestamp(dates[pos])
            previous_positions = []
            for prev_pos in range(pos - 1, -1, -1):
                prev_date = pd.Timestamp(dates[prev_pos])
                day_diff = (current_date - prev_date).days
                if day_diff > story_window_days:
                    break
                if day_diff < 0:
                    continue
                previous_positions.append(prev_pos)

            if not previous_positions:
                continue

            prev_indices = sorted_indices[previous_positions]
            sims = embeddings[prev_indices] @ embeddings[current_idx]
            matched_prev = prev_indices[sims >= story_threshold]

            for prev_idx in matched_prev:
                if dsu.union(int(prev_idx), int(current_idx)):
                    edge_count += 1

    root_to_cluster = {}
    cluster_ids = []
    next_cluster = 0

    for idx in range(len(news)):
        root = dsu.find(idx)
        if root not in root_to_cluster:
            root_to_cluster[root] = f"cluster_{next_cluster:06d}"
            next_cluster += 1
        cluster_ids.append(root_to_cluster[root])

    return pd.Series(cluster_ids, index=news_df.index, name="cluster_id"), edge_count

In [ ]:
def pairwise_clustering_metrics(reference_df, candidate_df, id_column="news_id", cluster_column="cluster_id"):
    ref = reference_df[[id_column, cluster_column]].copy()
    cand = candidate_df[[id_column, cluster_column]].copy()
    ref[id_column] = normalize_news_id(ref[id_column])
    cand[id_column] = normalize_news_id(cand[id_column])

    merged = ref.merge(cand, on=id_column, how="inner", suffixes=("_ref", "_cand"))
    n = len(merged)
    if n < 2:
        return {"n_items": n, "pairwise_precision": 0.0, "pairwise_recall": 0.0, "pairwise_f1": 0.0, "false_merge_rate": 0.0}

    ref_labels = merged[f"{cluster_column}_ref"].astype(str).to_numpy()
    cand_labels = merged[f"{cluster_column}_cand"].astype(str).to_numpy()

    tp = fp = fn = 0
    total_pairs = 0

    for i in range(n):
        ref_same = ref_labels[i] == ref_labels[i + 1:]
        cand_same = cand_labels[i] == cand_labels[i + 1:]
        tp += int(np.logical_and(ref_same, cand_same).sum())
        fp += int(np.logical_and(~ref_same, cand_same).sum())
        fn += int(np.logical_and(ref_same, ~cand_same).sum())
        total_pairs += len(ref_same)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    false_merge_rate = fp / (tp + fp) if (tp + fp) else 0.0

    return {
        "n_items": n,
        "total_pairs": total_pairs,
        "tp_same_pairs": tp,
        "fp_false_merge_pairs": fp,
        "fn_missed_same_pairs": fn,
        "pairwise_precision": precision,
        "pairwise_recall": recall,
        "pairwise_f1": f1,
        "false_merge_rate": false_merge_rate,
    }

In [ ]:
if not CANDIDATE_POOL_PATH.exists() or not GOLDEN_PATH.exists():
    print("Downstream files not found; skipping downstream smoke-test.")
else:
    candidate_pool_raw = pd.read_csv(CANDIDATE_POOL_PATH)
    candidate_pool = prepare_model_input(candidate_pool_raw)

    golden = pd.read_csv(GOLDEN_PATH)
    golden["news_id"] = normalize_news_id(golden["news_id"])

    base_downstream_cache = ARTIFACTS_DIR / "embeddings" / "bge_m3_candidate_pool_base_for_08.npz"
    ft_downstream_cache = ARTIFACTS_DIR / "embeddings" / "bge_m3_candidate_pool_finetuned_for_08.npz"

    base_candidate_embeddings = get_or_create_embeddings(
        base_model, candidate_pool, base_downstream_cache, batch_size=DOWNSTREAM_ENCODE_BATCH_SIZE
    )
    ft_candidate_embeddings = get_or_create_embeddings(
        ft_model, candidate_pool, ft_downstream_cache, batch_size=DOWNSTREAM_ENCODE_BATCH_SIZE
    )

    print("candidate_pool:", candidate_pool.shape)
    print("golden:", golden.shape)
    print("base embeddings:", base_candidate_embeddings.shape)
    print("ft embeddings:", ft_candidate_embeddings.shape)

In [ ]:
if CANDIDATE_POOL_PATH.exists() and GOLDEN_PATH.exists():
    threshold_values = [0.80, 0.81, 0.82, 0.83, 0.84, 0.85, 0.86]
    rows = []

    for model_name, embeddings in [
        ("base_bge_m3", base_candidate_embeddings),
        ("finetuned_bge_m3", ft_candidate_embeddings),
    ]:
        for threshold in threshold_values:
            cluster_ids, edge_count = threshold_graph_cluster(
                candidate_pool,
                embeddings,
                story_threshold=threshold,
                story_window_days=14,
                use_topic_blocking=True,
            )

            pred_clusters = candidate_pool[["news_id"]].copy()
            pred_clusters["cluster_id"] = cluster_ids.astype(str).values

            metrics = pairwise_clustering_metrics(
                reference_df=golden.dropna(subset=["cluster_id"]),
                candidate_df=pred_clusters,
            )

            rows.append({
                "model": model_name,
                "story_threshold": threshold,
                "edge_count": edge_count,
                "n_clusters": int(pd.Series(cluster_ids).nunique()),
                **metrics,
            })

    downstream_clustering_table = pd.DataFrame(rows).sort_values(
        ["model", "pairwise_f1", "false_merge_rate"],
        ascending=[True, False, True],
    ).reset_index(drop=True)

    display(downstream_clustering_table)

## 6. Как читать результат

1. `retrieval_table`: fine-tuned BGE-M3 должен не ухудшить поиск парафразов.
2. `downstream_clustering_table`: fine-tuned BGE-M3 не должен ухудшить `false_merge_rate`. Если `false_merge_rate` растёт, модель стала слишком агрессивно склеивать похожие, но разные сюжеты.
3. Если fine-tuned модель на каком-то `story_threshold` даёт выше `pairwise_f1` при низком `false_merge_rate`, тогда имеет смысл переносить её в `06_model_improvement_experiments.ipynb` и делать полноценный `exp_08` с переобучением final novelty classifier.

Если paraphrase retrieval улучшился, а downstream clustering ухудшился — это не победа для нашего проекта.